In [1]:
import pandas as pd


# Load CSV
number = 1200
#df = pd.read_csv(f'./Data_saved/kl_scheduling/allRows_vin2/Generated_data_0_1EVpenalty/Generated_data/epoch_{number}/EV_generated_data_{number}.csv')
#df = pd.read_csv(f'./Generated_data/epoch_{number}/EV_generated_data_{number}.csv')
#df = pd.read_csv(f'./Data_saved/kl_scheduling/allRows_vin2/Generated_data_0_1EVpenalty/Generated_data/EV_generated_data_cleaned.csv')
df = pd.read_csv(f'./Data_saved/kl_scheduling/allRows_vin1/EV_generated_data_cleaned.csv')
    
# Identify consecutive 'charge' groups
df['is_charge'] = (df['event'].eq('charge')) & (df['charge'].gt(0))

df['charge_group'] = (df['is_charge'] & ~df['is_charge'].shift(fill_value=False)).cumsum()
df.loc[~df['is_charge'], 'charge_group'] = None

# Aggregate consecutive charge sessions
sessions = (
    df.dropna(subset=['charge_group'])
      .groupby('charge_group')
      .agg(
          session_length=('event', 'count'),
          charge_modes=('charge_mode', lambda x: list(x)),
          unique_modes=('charge_mode', lambda x: set(x))
      )
)

# Filter only sessions with multiple consecutive charges
multi_sessions = sessions[sessions['session_length'] > 1]

print("All charging sessions:")
print(sessions)

print("\nMultiple consecutive charging sessions:")
print(multi_sessions)

# Count frequency of different types of multi sessions
counts = multi_sessions['unique_modes'].value_counts()
print("\nCounts of different charge_mode combinations:")
print(counts)


All charging sessions:
              session_length charge_modes unique_modes
charge_group                                          
1.0                        1        [120]        {120}
2.0                        2   [120, 120]        {120}
3.0                        1        [120]        {120}
4.0                        1        [120]        {120}
5.0                        1        [240]        {240}
...                      ...          ...          ...
1498.0                     1        [240]        {240}
1499.0                     1        [120]        {120}
1500.0                     1        [240]        {240}
1501.0                     1        [240]        {240}
1502.0                     1        [120]        {120}

[1502 rows x 3 columns]

Multiple consecutive charging sessions:
              session_length        charge_modes        unique_modes
charge_group                                                        
2.0                        2          [120, 120]          

## Per-window analysis

In [3]:
import pandas as pd

## Set SAVE = True if you'd like to save csv with all valid windows without consecutive charges
SAVE = False



# Load data
#data = pd.read_csv(f'./Data_saved/kl_scheduling/allRows_vin2/Generated_data_NO_shuffle/EV_generated_data_cleaned.csv')
data = pd.read_csv(f'./Data_saved/kl_scheduling/allRows_vin1/EV_generated_data_cleaned.csv')


# Define window size, minimum and maximum charge
seq_len = 10

# Split data into windows
windows = [data.iloc[i:i+seq_len].copy() for i in range(0, len(data), seq_len)]

# Initialize counters
total_windows = len(windows)
windows_with_same_mode = 0
windows_with_diff_mode = 0
windows_without_consecutive_charges = 0

# Lists to store results
valid_windows = []             # windows without anomalies
anomalous_windows = []         # windows with mixed charge modes

for i, df in enumerate(windows):
    
    # Fix charge_mode 0 → 120 for charge events
    df.loc[(df['event'] == 'charge') & (df['charge_mode'] == 0), 'charge_mode'] = 120
    
    # Identify charge events
    df['is_charge'] = (df['event'].eq('charge')) & (df['charge'].gt(0))
    

    # Identify consecutive charge groups
    df['charge_group'] = (df['is_charge'] & ~df['is_charge'].shift(fill_value=False)).cumsum()
    df.loc[~df['is_charge'], 'charge_group'] = None

    # Aggregate consecutive charge sessions
    sessions = (
        df.dropna(subset=['charge_group'])
          .groupby('charge_group')
          .agg(
              session_length=('event', 'count'),
              charge_modes=('charge_mode', list),
              unique_modes=('charge_mode', set)
          )
    )

    # Filter only sessions with multiple consecutive charges
    multi_sessions = sessions[sessions['session_length'] > 1]

    if multi_sessions.empty:
        windows_without_consecutive_charges += 1
        valid_windows.append(df)
        continue

    # Check if at least one multi-session has multiple different modes
    has_diff_modes = any(len(modes) > 1 for modes in multi_sessions['unique_modes'])

    if has_diff_modes:
        windows_with_diff_mode += 1
        anomalous_windows.append(df)
    else:
        windows_with_same_mode += 1
        valid_windows.append(df)

# === Summary Statistics ===
print("=== Consecutive Charge Analysis (per-window) ===")
print(f"Total windows: {total_windows}")
print(f"Windows without consecutive charges: {windows_without_consecutive_charges}")
print(f"Windows with consecutive charges (same charge_mode): {windows_with_same_mode}")
print(f"Windows with consecutive charges (different charge_modes): {windows_with_diff_mode}")

# Percentages
print("\n=== Percentages ===")
print(f"No consecutive charges: {windows_without_consecutive_charges / total_windows * 100:.2f}%")
print(f"Same mode consecutive charges: {windows_with_same_mode / total_windows * 100:.2f}%")
print(f"Different mode consecutive charges: {windows_with_diff_mode / total_windows * 100:.2f}%")

# === Save or inspect valid (non-anomalous) windows ===
print("\nNumber of valid (non-anomalous) windows:", len(valid_windows))

# Concatenate all valid windows into one DataFrame
valid_data = pd.concat(valid_windows, ignore_index=True)
valid_data.drop(columns=['is_charge', 'charge_group'], inplace=True)

# Save to CSV
if SAVE:
    valid_data.to_csv('./Data_saved/kl_scheduling/allRows_vin1/valid_data_no_penalty.csv', index=False)

=== Consecutive Charge Analysis (per-window) ===
Total windows: 943
Windows without consecutive charges: 702
Windows with consecutive charges (same charge_mode): 77
Windows with consecutive charges (different charge_modes): 164

=== Percentages ===
No consecutive charges: 74.44%
Same mode consecutive charges: 8.17%
Different mode consecutive charges: 17.39%

Number of valid (non-anomalous) windows: 779
